# Week 7 – Model Deployment & Real-World Application

## Introduction

In the previous weeks, I focused on preparing data, engineering features, building machine learning models, and evaluating their performance. However, a machine learning model becomes more useful when it can be deployed and accessed outside a Jupyter Notebook.

In Week 7, the Titanic Survival Prediction model developed in the previous weeks will be prepared for real-world use. The trained model will be saved for reuse and deployed through a Flask API that can accept passenger information and return a survival prediction.

A simple Streamlit web application will also be developed to provide a user-friendly interface where users can enter passenger information and receive predictions without interacting directly with Python code.

This project demonstrates the transition from model development to model deployment and provides practical experience in making machine learning models accessible to users and other applications.

## Objective

The objective of this project is to deploy a trained machine learning model and make it capable of generating predictions through an API and a simple web application.

The project will involve:

- Saving the trained machine learning model for future use.
- Building a prediction API using Flask.
- Testing the API with sample input data.
- Creating a Streamlit user interface for making predictions.
- Documenting the deployment process and providing instructions for running the application.

In [52]:
# Importing the libraries needed for model deployment and API testing.
import os
import joblib
import pandas as pd
import requests

In [53]:
# Checking that the saved model file is available.
model_path = "final_titanic_logistic_regression.pkl"

print("Model exists:", os.path.exists(model_path))

Model exists: True


In [54]:
# Loading the optimized Logistic Regression model.
model = joblib.load(model_path)

print(model)

LogisticRegression(max_iter=1000, random_state=42)


## Flask API

In [55]:
%%writefile app.py

from flask import Flask, request, jsonify
import joblib
import pandas as pd


# Creating the Flask application.
app = Flask(__name__)


# Loading the saved Logistic Regression model.
model = joblib.load("final_titanic_logistic_regression.pkl")


def preprocess_input(data):

    # Converting the input data into a DataFrame.
    df = pd.DataFrame([data])

    # Creating the FamilySize feature.
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

    # Identifying passengers travelling alone.
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    # Creating age groups from the passenger's age.
    df["AgeGroup"] = pd.cut(
        df["Age"],
        bins=[0, 12, 18, 35, 60, 100],
        labels=[
            "Child",
            "Teenager",
            "Young Adult",
            "Adult",
            "Senior"
        ]
    )

    # Converting categorical variables into numerical features.
    df = pd.get_dummies(
        df,
        columns=["Sex", "Embarked", "Title", "AgeGroup"],
        drop_first=True
    )

    # Defining the features required by the model.
    required_features = [
        "Pclass",
        "Age",
        "SibSp",
        "Parch",
        "Fare",
        "FamilySize",
        "IsAlone",
        "HasCabin",
        "Sex_male",
        "Embarked_Q",
        "Embarked_S",
        "Title_Miss",
        "Title_Mr",
        "Title_Mrs",
        "Title_Rare",
        "AgeGroup_Teenager",
        "AgeGroup_Young Adult",
        "AgeGroup_Adult",
        "AgeGroup_Senior"
    ]

    # Adding missing features with zero values.
    for feature in required_features:
        if feature not in df.columns:
            df[feature] = 0

    # Keeping the same feature order used during training.
    df = df[required_features]

    return df


@app.route("/predict", methods=["POST"])
def predict():

    # Receiving passenger information from the request.
    data = request.get_json()

    # Processing the passenger information.
    processed_data = preprocess_input(data)

    # Generating the survival prediction.
    prediction = model.predict(processed_data)[0]

    # Converting the prediction into a readable result.
    result = "Survived" if prediction == 1 else "Did not survive"

    # Returning the prediction as JSON.
    return jsonify({
        "prediction": int(prediction),
        "result": result
    })


# Running the Flask application locally.
if __name__ == "__main__":
    app.run(debug=True)

Overwriting app.py


## API Testing

In [56]:
# Starting the Flask API in the background.
import subprocess
import sys

flask_process = subprocess.Popen(
    [sys.executable, "app.py"]
)

print("Flask API is starting...")

Flask API is starting...


In [57]:
# Creating a sample passenger for API testing.
sample_passenger = {
    "Pclass": 3,
    "Sex": "male",
    "Age": 22,
    "SibSp": 1,
    "Parch": 0,
    "Fare": 7.25,
    "Embarked": "S",
    "Title": "Mr",
    "HasCabin": 0
}

In [58]:
# Sending the sample passenger to the prediction endpoint.
response = requests.post(
    "http://127.0.0.1:5000/predict",
    json=sample_passenger
)

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'prediction': 0, 'result': 'Did not survive'}


### API Result

The prediction API was tested using sample passenger information. The request was successfully processed and returned a status code of 200.

The model predicted that the sample passenger did not survive.

The successful response confirms that the API can receive passenger information, apply the required preprocessing, generate a prediction using the saved machine learning model, and return the result in JSON format.

## Streamlit App

In [59]:
%%writefile streamlit_app.py

import streamlit as st
import pandas as pd
import joblib

# I configured the Streamlit page for the Titanic prediction application.
st.set_page_config(
    page_title="Titanic Survival Predictor",
    page_icon="🚢"
)

# I displayed the title and a short description of the application.
st.title("🚢 Titanic Survival Predictor")
st.write("Enter the passenger information to generate a survival prediction.")

# I loaded the saved Logistic Regression model without retraining it.
model = joblib.load("final_titanic_logistic_regression.pkl")

# I collected the passenger class.
pclass = st.selectbox(
    "Passenger Class",
    [1, 2, 3]
)

# I collected the passenger's sex.
sex = st.selectbox(
    "Sex",
    ["male", "female"]
)

# I collected the passenger's age as a whole number.
age = st.number_input(
    "Age",
    min_value=0,
    max_value=100,
    value=25,
    step=1
)

# I collected the number of siblings or spouses.
sibsp = st.number_input(
    "Number of Siblings/Spouses",
    min_value=0,
    max_value=10,
    value=0,
    step=1
)

# I collected the number of parents or children.
parch = st.number_input(
    "Number of Parents/Children",
    min_value=0,
    max_value=10,
    value=0,
    step=1
)

# I collected the passenger fare.
fare = st.number_input(
    "Fare",
    min_value=0.0,
    value=30.0
)

# I collected the passenger's port of embarkation.
embarked = st.selectbox(
    "Port of Embarkation",
    ["S", "C", "Q"]
)

# I collected the passenger's title.
title = st.selectbox(
    "Title",
    ["Mr", "Miss", "Mrs", "Master", "Rare"]
)

# I collected whether cabin information was available.
has_cabin = st.selectbox(
    "Has Cabin",
    [0, 1]
)

# I generated a prediction when the user selected the prediction button.
if st.button("Predict Survival"):

    # I created a DataFrame containing the passenger information.
    passenger_data = pd.DataFrame([{
        "Pclass": pclass,
        "Sex": sex,
        "Age": age,
        "SibSp": sibsp,
        "Parch": parch,
        "Fare": fare,
        "Embarked": embarked,
        "Title": title,
        "HasCabin": has_cabin
    }])

    # I created the FamilySize feature used during model training.
    passenger_data["FamilySize"] = (
        passenger_data["SibSp"] +
        passenger_data["Parch"] +
        1
    )

    # I created the IsAlone feature.
    passenger_data["IsAlone"] = (
        passenger_data["FamilySize"] == 1
    ).astype(int)

    # I grouped the passenger's age into the same categories used during training.
    passenger_data["AgeGroup"] = pd.cut(
        passenger_data["Age"],
        bins=[0, 12, 18, 35, 60, 100],
        labels=[
            "Child",
            "Teenager",
            "Young Adult",
            "Adult",
            "Senior"
        ]
    )

    # I converted the categorical variables into numerical features.
    passenger_data = pd.get_dummies(
        passenger_data,
        columns=["Sex", "Embarked", "Title", "AgeGroup"],
        drop_first=True
    )

    # I defined the 19 features expected by the trained model.
    required_features = [
        "Pclass",
        "Age",
        "SibSp",
        "Parch",
        "Fare",
        "FamilySize",
        "IsAlone",
        "HasCabin",
        "Sex_male",
        "Embarked_Q",
        "Embarked_S",
        "Title_Miss",
        "Title_Mr",
        "Title_Mrs",
        "Title_Rare",
        "AgeGroup_Teenager",
        "AgeGroup_Young Adult",
        "AgeGroup_Adult",
        "AgeGroup_Senior"
    ]

    # I added missing features with a value of zero.
    for feature in required_features:
        if feature not in passenger_data.columns:
            passenger_data[feature] = 0

    # I arranged the features in the exact order expected by the model.
    passenger_data = passenger_data[required_features]

    # I generated the survival prediction using the saved model.
    prediction = model.predict(passenger_data)[0]

    # I displayed the prediction to the user.
    if prediction == 1:
        st.success("The model predicts that the passenger survived.")
    else:
        st.error("The model predicts that the passenger did not survive.")

Overwriting streamlit_app.py


In [60]:
# Starting the Streamlit application in the background.
streamlit_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "streamlit_app.py",
        "--server.headless",
        "true"
    ]
)

print("Streamlit application is starting...")

Streamlit application is starting...


## Streamlit Application Testing

#### Sample Passenger Profile

---
##### Profile 1

- Passenger Class: 3
- Sex: male
- Age: 22
- SibSp: 1
- Parch: 0
- Fare: 7.25
- Embarked: S
- Title: Mr
- Has Cabin: 0

##### Profile 2

- Passenger Class: 1
- Sex: female
- Age: 30
- SibSp: 0
- Parch: 0
- Fare: 100
- Embarked: S
- Title: Mrs
- Has Cabin: 1

The Streamlit interface was successfully connected to the Flask prediction API. Two different passenger profiles were entered through the web application to verify that the interface could send user inputs to the API and display the returned predictions.

The successful tests confirmed that the complete deployment workflow was functioning correctly:

Streamlit Interface → Flask API → Preprocessing → Logistic Regression Model → Prediction → Streamlit Result

In [61]:
readme_content = """# 🚢 AnalysLab Africa Week 7 — Model Deployment & Real-World Application

---

## 📌 Project Description

This project focuses on deploying a machine learning model developed to predict whether a passenger survived the Titanic disaster.

The model was developed using the Titanic dataset and trained to identify patterns between passenger characteristics and survival outcomes. After evaluating the models, the baseline Logistic Regression model with the highest accuracy was selected for deployment.

The trained model was saved using **Joblib** so that it could be reused without retraining. A **Flask API** was developed to receive passenger information and return survival predictions. A **Streamlit application** was also created to provide a simple interface for interacting with the prediction system.

The main objective of this project is to demonstrate how a machine learning model developed in a Jupyter Notebook can be converted into a usable application.

---

## ❓ Problem Statement

The Titanic dataset contains information about passengers who travelled on the RMS Titanic, including their age, sex, passenger class, family information, fare, embarkation point, and cabin information.

The objective of this project is to predict whether a passenger survived based on these characteristics.

The target variable is:
- `1` — Survived
- `0` — Did not survive

The model uses passenger information together with engineered features such as `FamilySize`, `IsAlone`, `Title`, `AgeGroup`, and `HasCabin`.

---

## 🤖 Model Used

The model used for deployment is a **Logistic Regression** classifier.

The baseline Logistic Regression model was selected because it achieved the highest accuracy among the models evaluated during the model development process.

### Model Configuration

```python
LogisticRegression(
    max_iter=1000,
    random_state=42
)
```

### Model Performance

The selected model achieved the following results before deployment:

| Metric    | Score  |
|-----------|--------|
| Accuracy  | 83.24% |
| Precision | 80.00% |
| Recall    | 75.36% |
| F1 Score  | 77.61% |
| ROC-AUC   | 87.58% |

The trained model was saved using Joblib as:

```
final_titanic_logistic_regression.pkl
```

The saved model can be loaded and reused without retraining.

---

## 🛠️ Technologies Used

The project was developed using:

| Technology       | Purpose                                  |
|------------------|------------------------------------------|
| Python           | Main programming language                |
| Pandas           | Data manipulation and preprocessing      |
| NumPy            | Numerical operations                     |
| Scikit-learn     | Machine learning                         |
| Joblib           | Saving and loading the trained model     |
| Flask            | Building the prediction API              |
| Streamlit        | Creating the web interface               |
| Requests         | Sending requests to the Flask API        |
| Jupyter Notebook | Development environment                  |

---

## 🔌 API Endpoints

The Flask application provides one prediction endpoint.

### `POST /predict`

This endpoint receives passenger information and returns the predicted survival outcome.

**Local endpoint:**
```
http://127.0.0.1:5000/predict
```

**Request method:** `POST`

### Input Format

The API accepts passenger information in JSON format.

**Example request:**

```json
{
    "Pclass": 3,
    "Sex": "male",
    "Age": 22,
    "SibSp": 1,
    "Parch": 0,
    "Fare": 7.25,
    "Embarked": "S",
    "Title": "Mr",
    "HasCabin": 0
}
```

### Input Variables

| Field    | Description                                        |
|----------|----------------------------------------------------|
| Pclass   | Passenger class                                    |
| Sex      | Passenger sex                                      |
| Age      | Passenger age                                      |
| SibSp    | Number of siblings or spouses aboard               |
| Parch    | Number of parents or children aboard               |
| Fare     | Passenger fare                                     |
| Embarked | Port of embarkation                                |
| Title    | Passenger title                                    |
| HasCabin | Indicates whether cabin information is available   |

The API processes these values and creates the required features before passing the data to the trained model.

### Output Format

The API returns a JSON response containing the predicted class and a readable result.

**Example response:**

```json
{
    "prediction": 0,
    "result": "Did not survive"
}
```

The prediction values represent:
- `1` — Survived
- `0` — Did not survive

A successful prediction request returns a status code of `200`.

---

## 🧪 API Testing

The Flask API was tested using sample passenger information.

A successful request returned a status code of `200`.

**Example response:**

```json
{
    "prediction": 0,
    "result": "Did not survive"
}
```

This confirmed that the API could successfully:
- Receive passenger information
- Process the input
- Load the saved model
- Generate a prediction
- Return the result

---

## 🖥️ Streamlit Application

A Streamlit application was created to provide a simple user interface for the prediction system.

The application allows users to enter passenger information such as:
- Passenger class
- Sex
- Age
- Number of siblings or spouses
- Number of parents or children
- Fare
- Port of embarkation
- Passenger title
- Cabin availability

The application sends the entered information to the Flask prediction API and displays the returned survival prediction.

> The age input is configured to accept whole numbers because passenger age is represented in years.

---

## 📁 Project Structure

```
Week 7/
│
├── Week_7_Model_Deployment.ipynb
├── app.py
├── streamlit_app.py
├── final_titanic_logistic_regression.pkl
└── README.md
```

### File Descriptions

| File                                      | Description                                                                 |
|-------------------------------------------|-----------------------------------------------------------------------------|
| `Week_7_Model_Deployment.ipynb`           | Contains the development and deployment process completed during Week 7     |
| `app.py`                                  | Contains the Flask application and prediction endpoint                      |
| `streamlit_app.py`                        | Contains the Streamlit user interface                                       |
| `final_titanic_logistic_regression.pkl`   | Contains the saved Logistic Regression model used to generate predictions   |
| `README.md`                               | Contains the project documentation and instructions for running the app     |

---

## ⚙️ Setup Instructions

### 1. Install Python

Python must be installed on the computer before running the project.

### 2. Install the Required Libraries

Run the following command in the terminal:

```bash
pip install pandas numpy scikit-learn joblib flask streamlit requests
```

### 3. Place the Project Files in the Same Directory

Make sure the following files are available in the project directory:
- `app.py`
- `streamlit_app.py`
- `final_titanic_logistic_regression.pkl`

> The saved model file is required because the Flask API loads it to generate predictions.

---

```
## ▶️ How to Run the Project

### Step 1 — Start the Streamlit Application

Open a terminal in the project directory and run:

```bash
streamlit run streamlit_app.py
```

Streamlit will provide a local URL, normally:

```
http://localhost:8501
```

Open the URL in a web browser.

### Step 2 — Enter Passenger Information

Enter the required passenger information through the Streamlit interface.

The application collects information such as:

- Passenger class
- Sex
- Age
- Number of siblings/spouses
- Number of parents/children
- Fare
- Port of embarkation
- Passenger title
- Cabin availability

### Step 3 — Generate a Prediction

Click the **Predict Survival** button.

The Streamlit application processes the passenger information, creates the required features, and passes the data directly to the saved Logistic Regression model.

The model then returns the predicted survival outcome, which is displayed in the application.

---

## 🌐 Flask API

The Flask API developed during this project can also be run separately for API testing.

To start the Flask API, open a terminal in the project directory and run:

```bash
python app.py
```

The Flask API will run locally at:

```
http://127.0.0.1:5000
```

The prediction endpoint is:

```
http://127.0.0.1:5000/predict
```

> The Flask API is included as part of the deployment work and can be used to test predictions through an API. However, the Streamlit application does not depend on the local Flask server and loads the saved model directly.

---

## 🔄 Deployment Workflow

The Streamlit application follows this workflow:

```
User
  ↓
Streamlit Application
  ↓
Input Processing
  ↓
Feature Engineering
  ↓
Saved Logistic Regression Model
  ↓
Prediction
  ↓
Streamlit Result
```

The Flask API follows a separate workflow for API-based testing:

```
User / API Request
  ↓
Flask /predict Endpoint
  ↓
Input Processing
  ↓
Feature Engineering
  ↓
Saved Logistic Regression Model
  ↓
Prediction
  ↓
JSON ResponseS
```
```


---

## ✅ Conclusion

The Titanic survival prediction model was successfully prepared for real-world use.

The baseline Logistic Regression model with the highest accuracy was saved using Joblib and integrated into a Flask API. The API was successfully tested and returned predictions with a status code of `200`.

A Streamlit application was also created to provide a simple interface through which users can enter passenger information and receive survival predictions.

This project demonstrates the transition from developing and evaluating a machine learning model in Jupyter Notebook to making the trained model accessible through an API and web application.
"""

# Write the README.md file
output_path = "README.md"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(readme_content)

print(f"README.md successfully generated at: {output_path}")
print(f"Total characters: {len(readme_content)}")
print(f"Total lines: {len(readme_content.splitlines())}")


README.md successfully generated at: README.md
Total characters: 10093
Total lines: 373
